In [10]:
# Optimisation du portefeuille 

# On possède 5 actions (ex: Apple, Microsoft, Google...)
# --> utiliser NumPy pour calculer la matrice de covariance de leurs rendements. 
# --> utiliser scipy.optimize.minimize pour trouver exactement le poids de chaque action 

# Objectif : maximiser le ratio de Sharpe (le meilleur rendement pour le risque le plus faible).

# Sharpe = (E(Rp) - Rf)/sigma
# --> E(Rp) rendement du portefeuille attendu
# --> Rf = taux sans risque (qu'on fixe à 0 pour simplifier)
# --> sigma = volatilité du portefeuille

In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from portfolio_optimizer import portfolio_statistics, maximize_sharpe_ratio

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# Utilisation du générateur recommandé pour isoler l'état aléatoire
rng = np.random.default_rng(seed=42)

assets_list = ['AAPL', 'MSFT', 'GOOG', 'AMZN']
n_assets = len(assets_list)

# Simulation de 5 ans de rendements journaliers (1260 jours de trading)
simulated_data = rng.normal(loc=0.0005, scale=0.015, size=(1260, n_assets))
returns_df = pd.DataFrame(simulated_data, columns=assets_list)

In [13]:
TRADING_DAYS_PER_YEAR = 252

# Étape A : Calcul des moyennes et covariances journalières, puis annualisation
annual_mean_returns = returns_df.mean().to_numpy() * TRADING_DAYS_PER_YEAR
annual_cov_matrix = returns_df.cov().to_numpy() * TRADING_DAYS_PER_YEAR

print("--- STATISTIQUES ANNUELLES CALCULÉES ---")
for asset, r in zip(assets_list, annual_mean_returns):
    print(f"  {asset} -> Rendement moyen attendu : {r*100:.2f}%")

--- STATISTIQUES ANNUELLES CALCULÉES ---
  AAPL -> Rendement moyen attendu : 10.76%
  MSFT -> Rendement moyen attendu : 7.46%
  GOOG -> Rendement moyen attendu : -2.91%
  AMZN -> Rendement moyen attendu : 1.78%


In [14]:
# Étape B : Lancement de l'optimiseur non linéaire
optimal_weights = maximize_sharpe_ratio(annual_mean_returns, annual_cov_matrix)

# Étape C : Calcul des métriques finales du portefeuille optimal
opt_return, opt_vol, opt_sharpe = portfolio_statistics(optimal_weights, annual_mean_returns, annual_cov_matrix)

print("=============================================")
print("=== ALLOCATION OPTIMALE DES POIDS ===")
print("=============================================")
for asset, weight in zip(assets_list, optimal_weights):
    print(f"  {asset} : {weight*100:.2f}%")

print("\n=============================================")
print("=== PERFORMANCES ATTENDUES DU PORTEFEUILLE ===")
print("=============================================")
print(f"Rendement annuel attendu     : {opt_return*100:.2f}%")
print(f"Volatilité annuelle attendue : {opt_vol*100:.2f}%")
print(f"Ratio de Sharpe Maximum      : {opt_sharpe:.2f}")
print("=============================================")

=== ALLOCATION OPTIMALE DES POIDS ===
  AAPL : 54.50%
  MSFT : 36.74%
  GOOG : 0.00%
  AMZN : 8.76%

=== PERFORMANCES ATTENDUES DU PORTEFEUILLE ===
Rendement annuel attendu     : 8.76%
Volatilité annuelle attendue : 15.60%
Ratio de Sharpe Maximum      : 0.56


In [ ]:
" Remarques "

# 1. Pourquoi maximiser le Ratio de Sharpe via l'inversion de signe ?
# L'algorithme `scipy.optimize.minimize` ne peut que minimiser
# Notre objectif financier est d'atteindre le point le plus haut du ratio de Sharpe (un maximum)
# --> minimiser la fonction négative 

# 2. Algèbre linéaire : L'élégance de l'opérateur `@` pour la Volatilité
# Formulation quadratique
# --> La variance d'un portefeuille dépend de la manière dont les actifs varient ensemble. 
# --> S'exprime par la formule matricielle W^T*Sigma*W où Sigma est la matrice de covariance et W le vecteur des poids. 
# --> Remplacer l'ancien enchaînement de fonctions `np.dot` par l'écriture `w.T @ cov_matrix @ w` colle fidèlement aux notations de la littérature académique et s'exécute de manière hautement vectorisée 

# 3. Les limites pratiques du modèle en environnement réel
# Instabilité des paramètres :
# --> L'optimisation classique de Markowitz est extrêmement sensible aux erreurs d'estimation des rendements attendus (`mean_returns`). 
# En production réelle, de légères variations dans les prévisions de rendement peuvent induire des allocations aberrantes ou ultra-concentrées.
# --> C'est pourquoi les quants y ajoutent des contraintes plus strictes de diversification ou préfèrent utiliser des approches robustes comme le modèle de Black-Litterman ou l'optimisation de la Variance Minimale (qui ne dépend pas des rendements attendus).